load the data

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_parquet("../data/clean/traffic_violations_clean.parquet")
print(df.shape)
df["violation_category"].value_counts(normalize=True).round(3)

(2067761, 48)


violation_category
TRAFFIC SIGNAL / ROAD RULES    0.223
REGISTRATION                   0.178
SPEEDING                       0.174
LICENSE                        0.130
LIGHTS / EQUIPMENT             0.093
OTHER                          0.064
ALCOHOL / DUI                  0.036
PHONE USE                      0.027
RECKLESS DRIVING               0.022
WINDOW TINT                    0.019
SEAT BELT                      0.017
INSURANCE                      0.012
PARKING                        0.005
Name: proportion, dtype: float64

Q1: most common violations

the share of the top category, and which categories are small.

In [2]:
print(df["violation_category"].value_counts())
print(df["description"].value_counts().head(5))
px.bar(df["violation_category"].value_counts(), title="Violations by category").show()

violation_category
TRAFFIC SIGNAL / ROAD RULES    460107
REGISTRATION                   367135
SPEEDING                       360696
LICENSE                        268463
LIGHTS / EQUIPMENT             192190
OTHER                          131790
ALCOHOL / DUI                   74664
PHONE USE                       56781
RECKLESS DRIVING                44743
WINDOW TINT                     39884
SEAT BELT                       35026
INSURANCE                       25695
PARKING                         10587
Name: count, dtype: int64
description
DRIVER FAILURE TO OBEY PROPERLY PLACED TRAFFIC CONTROL DEVICE INSTRUCTIONS                   171274
FAILURE TO DISPLAY REGISTRATION CARD UPON DEMAND BY POLICE OFFICER                            91000
DRIVING VEHICLE ON HIGHWAY WITH SUSPENDED REGISTRATION                                        60761
DISPLAYING EXPIRED REGISTRATION PLATE ISSUED BY ANY STATE                                     58408
FAILURE OF INDIVIDUAL DRIVING ON HIGHWAY TO DISPL

Q2: which areas have the most incidents

which districts lead, and the missing-coordinates limitation.

In [3]:
print(df["sub_agency"].value_counts().head(5))
print("Missing coordinates %:", round(df["latitude"].isna().mean() * 100, 1))
px.bar(df["sub_agency"].value_counts().head(8), title="Top police districts").show()

sub_agency
4TH DISTRICT, WHEATON                              448580
3RD DISTRICT, SILVER SPRING                        373052
2ND DISTRICT, BETHESDA                             325179
6TH DISTRICT, GAITHERSBURG / MONTGOMERY VILLAGE    261210
5TH DISTRICT, GERMANTOWN                           245529
Name: count, dtype: int64
Missing coordinates %: 7.5


Q3: demographics vs violation type

the busiest hours and weekdays, and the sharp drop around 2020 (it was visible in your dashboard).

In [4]:
race_table = pd.crosstab(df["race"], df["violation_category"], normalize="index") * 100
print(race_table.round(1))
print(df["gender"].value_counts(normalize=True).round(3))

violation_category  ALCOHOL / DUI  INSURANCE  LICENSE  LIGHTS / EQUIPMENT  \
race                                                                        
ASIAN                         2.2        0.8      6.7                10.2   
BLACK                         2.9        1.7     16.3                10.0   
HISPANIC                      5.9        1.4     16.8                10.1   
NATIVE AMERICAN               2.4        1.0     12.8                10.7   
OTHER                         1.4        0.8      8.5                 8.4   
WHITE                         3.4        0.9      9.3                 8.1   

violation_category  OTHER  PARKING  PHONE USE  RECKLESS DRIVING  REGISTRATION  \
race                                                                            
ASIAN                 5.5      0.4        3.0               1.9          16.7   
BLACK                 5.9      0.4        2.2               1.9          20.8   
HISPANIC              8.0      0.5        2.2              

Q4: time patterns

the busiest hours and weekdays, and the sharp drop around 2020 (it was visible in your dashboard).

In [5]:
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday",
            "Friday", "Saturday", "Sunday"]
px.bar(df["stop_hour"].value_counts().sort_index(), title="By hour").show()
px.bar(df["stop_weekday"].value_counts().reindex(weekdays), title="By weekday").show()
px.bar(df["stop_year"].value_counts().sort_index(), title="By year").show()

Q5 and Q6: vehicles and accidents

which violation types have the highest accident rate, and the overall accident percentage.

In [6]:
print(df["make"].value_counts().head(5))
print(df["vehicle_type"].value_counts().head(3))

print("Accident %:", round(df["accident"].mean() * 100, 2))
print(df["severity"].value_counts())
accident_rate = df.groupby("violation_category", observed=True)["accident"].mean() * 100
px.bar(accident_rate.round(2).sort_values(ascending=False),
       title="% of violations involving an accident").show()

make
TOYOTA       356952
HONDA        307303
FORD         187504
NISSAN       153577
CHEVROLET    148273
Name: count, dtype: int64
vehicle_type
02 - AUTOMOBILE          1833630
05 - LIGHT DUTY TRUCK     108173
28 - OTHER                 35055
Name: count, dtype: int64
Accident %: 2.73
severity
NONE               1996948
PROPERTY DAMAGE      45137
INJURY               25109
FATAL                  567
Name: count, dtype: int64
